> Testing content_filter.py to ensure it can execute w/o error!

In [1]:
import os
#Ensure directory is set to /home/marqezorr/MyTop5
ROOT_DIR = os.path.join("/home", "marqezorr", "MyTop5")
os.chdir(ROOT_DIR)
print("Current directory:", os.getcwd()) #Ensure we're in the MyTop5 directory!

Current directory: /home/marqezorr/MyTop5


In [2]:
import pickle
from src.content_filter import get_content_recommendations #For testing the content filter system
from src.collab_filter import get_collab_recommendations #For testing the collaborative filter system 
from src.hybrid import get_hybrid_recommendations #For testing the hybrid system

## TESTING: get_content_recommendations

In [3]:
#load the pkl files produced by preprocess.py
with open("models/anime_metadata.pkl", "rb") as f:
    anime_df = pickle.load(f)
with open("models/content_feature_matrix.pkl", "rb") as f:
    feature_matrix = pickle.load(f)
with open("models/anime_index_map.pkl", "rb") as f:
    anime_index_map = pickle.load(f)

In [4]:
#Test content_filter with a few titles
selected_titles = [
    "Cowboy Bebop",
    "Fullmetal Alchemist",
    "Bleach",
    "Toradora!",
    "Arakawa Under the Bridge"
]

content_results = get_content_recommendations(
    selected_titles=selected_titles,
    anime_df=anime_df,
    feature_matrix=feature_matrix,
    anime_index_map=anime_index_map,
    top_n=25
)

print(content_results)


    anime_id                                               name  content_score
0       5114                   Fullmetal Alchemist: Brotherhood       0.746422
1       9135      Fullmetal Alchemist: The Sacred Star of Milos       0.735284
2       3036                                        Tobe! Isami       0.734900
3        635              Juubee-chan: Lovely Gantai no Himitsu       0.733904
4       1922           Urusei Yatsura Movie 3: Remember My Love       0.727995
5      31933  JoJo no Kimyou na Bouken Part 4: Diamond wa Ku...       0.727768
6       4106                            Trigun: Badlands Rumble       0.727317
7       1921          Urusei Yatsura Movie 2: Beautiful Dreamer       0.726234
8       1132                                   Digimon Frontier       0.725860
9      31490                               One Piece Film: Gold       0.725378
10      1923            Urusei Yatsura Movie 4: Lum The Forever       0.725070
11     33338                           One Piece: He

> The 5 anime I input are a mix of action-adventure, comedy, and romance; the output recommendations appear to match this taste fairly well!

## TESTING: get_collab_recommendations

In [5]:
#we already have anime_df, so only need svd_model loaded as well
with open("models/svd_model.pkl", "rb") as f:
    svd_model = pickle.load(f)

In [6]:
# Get all raw anime_ids that exist in the SVD trainset
all_inner_ids = list(svd_model.trainset._raw2inner_id_items.keys())

# Convert from Surprise's internal string format back to integers
all_trained_ids = sorted([int(raw_id) for raw_id in all_inner_ids])

print(f"Total anime in SVD trainset: {len(all_trained_ids)}")
print(f"Sample of IDs: {all_trained_ids[:10]}")

Total anime in SVD trainset: 9108
Sample of IDs: [1, 5, 6, 7, 8, 15, 16, 17, 18, 19]


In [7]:
collab_results = get_collab_recommendations(
    selected_titles=selected_titles,
    anime_df=anime_df,
    svd_model=svd_model,
    top_n=25
)

print(collab_results)

    anime_id                                               name  collab_score
0       5114                   Fullmetal Alchemist: Brotherhood      9.318136
1        820                               Ginga Eiyuu Densetsu      9.181015
2       9969                                           Gintama'      9.081956
3      28977                                           Gintama°      9.063435
4       9253                                        Steins;Gate      9.036107
5      11061                             Hunter x Hunter (2011)      8.985312
6        918                                            Gintama      8.968097
7       4181                               Clannad: After Story      8.952790
8       2001                         Tengen Toppa Gurren Lagann      8.936951
9      15417                                Gintama': Enchousen      8.917986
10       199                      Sen to Chihiro no Kamikakushi      8.888155
11      2904                 Code Geass: Hangyaku no Lelouch R2 

> The top 5 here includes an action-adventure series (FMA: Brotherhood), but it interestingly also includes **sci-fi** such as Ginga Eiyuu Densetsu & Steins;Gate, despite *none* of my input 5 titles being sci-fi. This does in fact indicate that the collaborative filtering system is introducing more genre/content diversity than the content filter does.

## TESTING: get_hybrid_recommendations

In [9]:
hybrid_results = get_hybrid_recommendations(
    selected_titles=selected_titles,
    anime_df=anime_df,
    feature_matrix=feature_matrix,
    anime_index_map=anime_index_map,
    svd_model=svd_model,
    top_n=10,
)

hybrid_results.head(10)

Generating content-based candidates...
Now generating collaborative filtering candidates...
Both candidate pools have been generated! # of candidates in the merged pool: 584
All set! Returning the top 10 hybrid recommendations now.


,anime_id,name,content_score,collab_score,hybrid_score,genres,synopsis
0,5114,Fullmetal Alchemist: Brotherhood,1.000000,1.000000,1.000000,"Action, Military, Adventure, Comedy, Drama, Ma...","""In order for something to be obtained, someth..."
1,820,Ginga Eiyuu Densetsu,0.441500,0.881961,0.639707,"Military, Sci-Fi, Space, Drama",The 150-year-long stalemate between the two in...
2,21,One Piece,0.805086,0.431876,0.637142,"Action, Adventure, Comedy, Super Power, Drama,...","Gol D. Roger was known as the ""Pirate King,"" t..."
3,6,Trigun,0.847779,0.327835,0.613804,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen","Vash the Stampede is the man with a $$60,000,0..."
4,9135,Fullmetal Alchemist: The Sacred Star of Milos,0.923157,0.215036,0.604502,"Action, Adventure, Comedy, Drama, Fantasy, Mag...",Chasing a runaway alchemist with strange power...
5,3036,Tobe! Isami,0.920510,0.215036,0.603047,"Action, Adventure, Comedy, Romance, Shounen",Isami Hanaoka is just a fifth grader who happe...
6,9969,Gintama',0.441500,0.796687,0.601334,"Action, Sci-Fi, Comedy, Historical, Parody, Sa...","fter a one-year hiatus, Shinpachi Shimura retu..."
7,635,Juubee-chan: Lovely Gantai no Himitsu,0.913638,0.215036,0.599267,"Action, Adventure, Comedy, Drama, Shounen",Jiyu Nanohana is an ordinary schoolgirl until ...
8,28977,Gintama°,0.441500,0.780743,0.594159,"Action, Comedy, Historical, Parody, Samurai, S...","Gintoki, Shinpachi, and Kagura return as the f..."
9,9253,Steins;Gate,0.441500,0.757218,0.583573,"Thriller, Sci-Fi",The self-proclaimed mad scientist Rintarou Oka...


> The hybrid system was initially biased towards anime with high CONTENT scores, as the merged candidate list was INNER joined; anime with high **collab_scores** often have low/no **content_scores**, and vice-versa, preventing them from hitting the merged candidate list despite being otherwise valuable.
> To address this, the merged candidate list was updated to form via OUTER join instead, allowing titles with high scores in **either** system to be included. Additionally, their missing score is imputed to the mean content/collab score to avoid these titles being harshly punished for being present in one result set but not the other.
> The end result is a recommendation list that far more closely resembles a true "fusion" between the content-based filter & the collaborative one.